# 02 - Scanned PDF to Content Understanding: original versus rendered
Start here. The input is always a scanned PDF. Send the original PDF to the configured CU analyzer, then render the same pages to a 300 dpi grayscale TIFF and send that derivative to the exact same analyzer. Compare the returned results.

PDFium only renders the scan; it does not perform OCR or extract business fields. Content Understanding remains the extraction service in both paths. No separate intake image or notebook 01 is required.

The original remains unchanged. This notebook reads only `demo/ocr/.env`. The same endpoint, analyzer and API version are used in both paths. Rendering does not recover missing source detail or guarantee an extraction improvement.

In [ ]:
import hashlib, json
from pathlib import Path
import pandas as pd
import pypdfium2 as pdfium
from PIL import Image
from dotenv import dotenv_values

OCR = Path('..').resolve()
if not (OCR / '.env').is_file():
    raise FileNotFoundError('Copy demo/ocr/.env.example to demo/ocr/.env first.')
config = dotenv_values(OCR / '.env')
PDF = (OCR / config['OCR_INPUT_FILE']).resolve()
DPI = int(config.get('OCR_TARGET_DPI', 300))
PAGE = int(config.get('OCR_FOCUS_PAGE', 1))
ANALYZER = (config.get('CU_ANALYZER_ID') or '').strip()
API = config.get('CU_API_VERSION') or '2025-11-01'
OUT = OCR / 'output'
(OUT / 'versions').mkdir(parents=True, exist_ok=True)
(OUT / 'screenshots').mkdir(exist_ok=True)
original_hash = hashlib.sha256(PDF.read_bytes()).hexdigest()
print('Input scanned PDF:', PDF)
print('Output directory:', OUT)

## Render all visible layers - local operation
Adapted from the existing [PDFium renderer and Pillow multi-page TIFF save](https://github.com/jomedinagomez/azure-ai-foundry-insurance/blob/f6949572c5aaa6bc77b1ac592a92affc62d98346/demo/sov/preprocess/pdf_render.py). Changes: configurable 300 dpi grayscale, no crop, explicit cleanup. [PDFium API](https://pypdfium2.readthedocs.io/en/stable/python_api.html) and [Pillow TIFF documentation](https://pillow.readthedocs.io/en/stable/handbook/image-file-formats.html#tiff).

Save only lossless TIFF/PNG. This example retains all pages and raises an explicit error if asked to save JPEG. The generated full-page TIFF is a new image, not the original fax stream.

In [ ]:
def save_lossless(pages, path, dpi):
    if path.suffix.casefold() not in ('.tif', '.tiff', '.png'):
        raise ValueError('Use TIFF or PNG; JPEG output is intentionally refused.')
    if path.suffix.casefold() == '.png' and len(pages) != 1:
        raise ValueError('Use a multi-page TIFF to preserve all pages.')
    if path.exists():
        with Image.open(path) as saved:
            identical = saved.n_frames == len(pages)
            if identical:
                for index, page in enumerate(pages):
                    saved.seek(index)
                    if saved.mode != page.mode or saved.size != page.size or saved.tobytes() != page.tobytes() or tuple(saved.info.get('dpi', ())) != tuple(dpi):
                        identical = False
                        break
            if identical:
                return
    compression = 'group4' if all(page.mode == '1' for page in pages) else 'tiff_lzw'
    options = {'compression': compression} if path.suffix.casefold() != '.png' else {}
    pages[0].save(path, save_all=len(pages) > 1, append_images=pages[1:], dpi=dpi, **options)

def render_pages(dpi):
    pages = []
    with pdfium.PdfDocument(str(PDF)) as document:
        for page in document:
            bitmap = page.render(scale=dpi / 72)
            try:
                pages.append(bitmap.to_pil().convert('L'))
            finally:
                bitmap.close()
    return pages

rendered = render_pages(DPI)
TIFF = OUT / 'versions' / f'rendered_{DPI}dpi.tiff'
save_lossless(rendered, TIFF, (DPI, DPI))
print('Full-page render:', len(rendered), 'pages | dimensions:', rendered[0].size, '| DPI:', DPI)
print('Uncropped multi-page TIFF:', TIFF.resolve())

## Preview the whole page
Show the complete selected page from the original scanned PDF and the saved TIFF, with no crop coordinates. `OCR_FOCUS_PAGE` selects a page; it does not select a region. The 150 dpi view is for display only. Both files sent to CU retain every full page. The absolute PDF, TIFF and screenshot paths are printed below.

In [ ]:
import matplotlib.pyplot as plt

control = render_pages(150)
if not 1 <= PAGE <= len(control):
    raise ValueError(f'OCR_FOCUS_PAGE must be between 1 and {len(control)} for this PDF.')
with Image.open(TIFF) as saved_tiff:
    saved_tiff.seek(PAGE - 1)
    processed_page = saved_tiff.copy()
views = [(f'Original PDF - full page {PAGE}', control[PAGE - 1]), (f'Processed TIFF - full page {PAGE}, {DPI} dpi', processed_page)]
fig, axes = plt.subplots(1, len(views), figsize=(12, 10))
for axis, (title, page) in zip(axes, views):
    axis.imshow(page, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
    axis.set_title(title)
    axis.axis('off')
fig.tight_layout()
preview_file = OUT / 'screenshots' / 'render_comparison.png'
fig.savefig(preview_file, dpi=160)
plt.show()
processed_page.close()
print('Original scanned PDF:', PDF.resolve())
print('Uncropped multi-page TIFF:', TIFF.resolve())
print('Full-page comparison image:', preview_file.resolve())

## Send both versions to the same Content Understanding analyzer
Adapted from the official [binary-analysis SDK sample](https://github.com/Azure/azure-sdk-for-python/blob/53023a2cd9adf255cff520a99985fff84dc58d7d/sdk/contentunderstanding/azure-ai-contentunderstanding/samples/sample_analyze_binary.py). [MIT notice](../LICENSE). `CU_ANALYZER_ID` selects the same existing analyzer for both inputs. Use the actual field-extraction analyzer to evaluate business fields; `prebuilt-layout` returns OCR/layout only. This notebook does not invent or replace an extraction schema. Cache hits reuse responses; cache misses make billable calls.

In [ ]:
from azure.ai.contentunderstanding import ContentUnderstandingClient
from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential

endpoint = config.get('CONTENTUNDERSTANDING_ENDPOINT', '').rstrip('/')
if endpoint and not ANALYZER:
    raise ValueError('Set CU_ANALYZER_ID in demo/ocr/.env to the existing analyzer used for extraction.')
key = config.get('CONTENTUNDERSTANDING_KEY')
client = ContentUnderstandingClient(endpoint=endpoint, credential=AzureKeyCredential(key) if key else DefaultAzureCredential(process_timeout=60), api_version=API) if endpoint else None

def analyze(path, label):
    if client is None:
        print('Skipped: set CONTENTUNDERSTANDING_ENDPOINT in demo/ocr/.env')
        return None
    data = path.read_bytes()
    digest = hashlib.sha256(data + f'{endpoint}|{ANALYZER}|{API}'.encode()).hexdigest()
    cache = OUT / 'cu-cache' / f'{digest}.json'
    cache.parent.mkdir(exist_ok=True)
    if cache.exists():
        result = json.loads(cache.read_text(encoding='utf-8'))
        print(label, '- cached')
    else:
        print(label, '- submitting to', ANALYZER)
        result = client.begin_analyze_binary(analyzer_id=ANALYZER, binary_input=data).result().as_dict()
        cache.write_text(json.dumps(result, indent=2), encoding='utf-8')
    (OUT / f'{label}.md').write_text('\n\n'.join(item.get('markdown', '') for item in result['contents']), encoding='utf-8')
    (OUT / f'{label}.input.json').write_text(json.dumps({'input_sha256': hashlib.sha256(data).hexdigest(), 'cache': cache.name, 'analyzer': ANALYZER, 'api_version': API}, indent=2), encoding='utf-8')
    return result

responses = {'original_pdf': analyze(PDF, 'original_pdf'), f'rendered_{DPI}dpi': analyze(TIFF, f'rendered_{DPI}dpi')}

## Takeaways
Compare the returned CU results, especially the affected amount and sign. Notebook 04 shows returned structured fields when the configured analyzer supplies them, and checks amounts in CU's Markdown separately. The input is the same scan in both cases; only preprocessing changed. A passing original does not demonstrate that preprocessing fixed a failure.

In [ ]:
keyword = config.get('OCR_KEYWORD', 'property tax').casefold()
for label, result in responses.items():
    print('\n', label)
    print('Input file:', (PDF if label == 'original_pdf' else TIFF).resolve())
    if result is None:
        print('Not analyzed.')
        continue
    metadata_file = OUT / f'{label}.input.json'
    evidence = json.loads(metadata_file.read_text(encoding='utf-8'))
    print('Full CU response JSON:', (OUT / 'cu-cache' / evidence['cache']).resolve())
    print('Full CU Markdown:', (OUT / f'{label}.md').resolve())
    print('Input provenance:', metadata_file.resolve())
    lines = '\n'.join(item.get('markdown', '') for item in result['contents']).splitlines()
    matches = [i for i, line in enumerate(lines) if keyword in line.casefold()]
    if not matches:
        print('Label not found; open the full CU Markdown file printed above.')
    for i in matches[:2]:
        print('\n'.join(lines[max(0, i - 1):i + 3]))
for pages in (control, rendered):
    for page in pages:
        page.close()
print('Original unchanged:', hashlib.sha256(PDF.read_bytes()).hexdigest() == original_hash)